# 4.2 OOD Detection y PCA Visualization en Weka
Uso del intérprete para evaluar la capacidad de las medidas de incertidumbre para separar datos conocidos (Visible) de datos desconocidos (Shadow). Además, visualizamos el efecto del clustering mediante una proyección PCA.

In [ ]:
import os
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.generators import ARFFDataGenerator
from src.shadows import ClosestClusterShadow
from src.splitter import GapSplitter
from src.orchestrator import RegressionOrchestator
from src.models import RandomForestModel, NeuralNetworkMCDropoutModel, BayesianNeuralNetworkModel
from src.ensemble import EnsembleRegressor
from src.noise import AbsoluteGaussianNoise
from src.interpreter import UncertaintyInterpreter
from src.visualization import plot_weka_pca


In [ ]:
datasets = ['bodyfat', 'cpu_act', 'housing', 'pbc', 'quake', 'cal_housing', 'house_16H', 'mv']
data_dir = '../data/weka_arff'

m_rf = RandomForestModel(n_estimators=50, max_depth=10)
m_drop = NeuralNetworkMCDropoutModel(hidden_dim=64, dropout_rate=0.2, epochs=300, n_iter=50)
m_bnn = BayesianNeuralNetworkModel(hidden_dim=32, epochs=300, n_iter=50)
ensemble = EnsembleRegressor(models=[m_rf, m_drop, m_bnn])

models = [m_rf, m_drop, m_bnn, ensemble]
shadow = ClosestClusterShadow(k_range=(5, 10), random_state=42)
splitter = GapSplitter(test_size=0.2, random_state=42)
dummy_noise = AbsoluteGaussianNoise(std_dev=0.0)

In [ ]:
results = []
import warnings
warnings.filterwarnings('ignore')

for dataset_name in datasets:
    print(f"\n{'='*50}")
    print(f"Datu-multzoa prozesatzen: {dataset_name}")
    print(f"{'='*50}")
    
    generator = ARFFDataGenerator(dataset_name=dataset_name, data_dir=data_dir)
    
    dummy_orch = RegressionOrchestator(generator=generator, splitter=splitter, model=m_rf)
    artifacts = dummy_orch.run(n_samples=-1, x_range=(0,0), noise_strategy=dummy_noise, shadow=shadow)
    
    plot_weka_pca(artifacts.split_data, title=f"PCA Proyección - {dataset_name}")
    
    for model in models:
        print(f"  [Eval] {model.get_name()}")
        orchestrator = RegressionOrchestator(generator=generator, splitter=splitter, model=model)
        
        try:
            art = orchestrator.run(n_samples=-1, x_range=(0,0), noise_strategy=dummy_noise, shadow=shadow)
            
            unc_vis = art.uncertainties['visible']
            unc_shadow = art.uncertainties['shadow']
            
            interpreter = UncertaintyInterpreter()
            eval_metrics = interpreter.evaluate(unc_vis, unc_shadow)
            
            results.append({
                'Dataset': dataset_name,
                'Model': model.get_name(),
                'OOD Detection Acc': eval_metrics['accuracy'],
                'Known Acc': eval_metrics['acc_known'],
                'Shadow Acc': eval_metrics['acc_unknown']
            })
        except Exception as e:
            print(f"  [ERROR] {e}")

df_interpreter = pd.DataFrame(results)
display(df_interpreter)

In [ ]:
if not df_interpreter.empty:
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_interpreter, x='Dataset', y='OOD Detection Acc', hue='Model')
    plt.title('Precisión (Accuracy) detectando datos de las Sombras vs Conocidos usando Incertidumbre')
    plt.axhline(0.5, color='black', linestyle='--', label='Azar (50%)')
    plt.xticks(rotation=45)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()